<h1 style="text-align:center;">Quantifying Acne Severity Dynamics - A Hierarchical Bayesian Approach</h1>
<p style="text-align:center;">By Nathaniel Wolff.</p>



# Overview

The following notebook describes the structure of a Machine Learning model/simulator that predicts acne patient severity trajectories, given a series of treatments at varying dosages and daily meal data logged by the patient. 



### Analysis Steps
#### 1) Data Parsing -
Base dataframe is seperated by patient. Raw acne severities are normalized as % change relative to patient average baseline intensity. 

# Model Structure and Parameter Estimation


### Latent State Space/Hieararchical Bayesian Model for Latent State to Severity Mapping

Here, Acne Severity is modeled as a random variable $S_{t}$, distributed according to the following Bayesian Hieararchial Model, smoothed with a Kalman Filter. Here, acne severity represents the inherent patholophysiology of acne of all patients, meaning there is low variance
in this measure for all patients regardless of genetic background. 

#### Layer 1: 
The first layer embeds a raw data vector of easily measurable patient diagnostics in acne pathology into the appropriate feature space 
on the distribution of clinically practical diagnostics, $\hat{o} = \begin{bmatrix} pH_{skin} & [Insulin] & [IGF1] & NLR\end{bmatrix}$. These raw diagnostic biomarkers are embedded in a feature space, defined by the following features modeling important contributors to acne pathology, per biochemical and immunological mechanisms:

Bacterial Dysbiosis Lipase Effect, $L_{t}$: $\sum_{i=1}^{n} \frac{1}{\sigma_{i}\sqrt{2\pi}} \exp\left(-\frac{(pH_{skin} - \mu_{i})^2)}{2\sigma^2_{i}}\right)$ (to be later updated with kinetic modeling approaches). 

mTORC1 Effect, $MT_{T}$: $\alpha_{1} * [Insulin]/([Insulin] + (K_{d}))_{Insulin} + \alpha_{2} * [IFG1]/([IFG1] + (K_{d}))_{IFG1}$. 

Partial Inflammatory Effect, $I_{t}$: $\alpha_{3} * NLR$

#### Layer 2: 
This vector of these effects, $\hat{o'}$, is mapped to a vector of imputed biomarkers via the linear transformation W, and serves as the mean for the distribution of imputed biomarkers, $\hat{Y_{t}}$: $\hat{Y_{t}} = \begin{bmatrix} Lipase\,Effect \\ [mTORC1] \\ Inflammam \end{bmatrix}$, which is distributed normallly: $\hat{Y_{t}} \sim \mathcal{N}(W\hat{o'},\,\Sigma_{1})\,$. This is the layer that the following Nonlinear Mixed Effects Model enters, via the matrix of treatments $\hat T_{t} = \begin{bmatrix} [BPO-Clin]_{1}, \, \, ... \, \, [BPO-Clin]_{t}]\\ [Isotretinoin]_{1} \, \, ... \, \, [Isotretinoin]_{t} \end{bmatrix}$ that patients respond to differently due to variation in diet. The $1, j$ element of said matrix is the concentration of antibiotics taken by the patient on that day, while the $2, k$ element of said matrix is the concentration of retinol taken.    

#### Layer 3: 
In turn, $\hat{Y_{t}}$ is mapped to a full latent state vector, $\hat{Z_{t}}$, defined the same way as the data-driven approach: $\hat{Z_{t}} = \begin{bmatrix} B_{t} \\ I_{t} \\ S_{t} \end{bmatrix}$. The mapping, via linear transformation A, defines the latent state's own normal distribution: $\hat{Z_{t}} \sim \mathcal{N}(\hat{AY_{t}} + OU,\,\Sigma_{2})\,$, where $OU$ is a stochastic contribution from the Ornstein-Uhlenbeck process governing the reversion of the latent states to the respective mean, as a result of other biological pathways not explicitly modeled here.

#### Layer 4:
The final layer determines the distribution of acne severity, $S_{t}$, as a function of $\hat{Y_{t}}$. Here, acne severity is imputed to follow a Normal - Gamma distribution, such that the maximized likelihood from the first 3 layers is conjugate to the Gamma prior. As such,

$S_{t} \sim \Gamma(\alpha_{n}, \beta_{n})$ where $ \alpha_{n} / \beta_{n} = exp(W \cdot Y_{t} + b)$, with learned weighting vector w and bias vector b.$ 



### Nonlinear Mixed Effect/Gradient Boosting Regression Tree Model for Patient-Specific Treatment Response


A patient's imputed biomarker vector, given a particular treatment series, is denoted as $ Y_{TS}= Y \mid \hat T_{t}$. The mapping to this vector from from the treatment series, denoted as $V(\phi_{i,j}, \, \hat T_{t})$, has a domain corresponding to patient specific parameter vectors. The space corresponding to the domain is interpreted as the set of patient-specific parameter vectors describing responsiveness to treatment. So, $Y_{TS}$ is modeled with a nonlinear mixed effect model,$(Y_{TS})_{i,j} = V(\phi_{i,j}, \, \hat T_{j}) + \epsilon_{i,j}$. In this case, $\phi_{i,j}$ is the diet covariate tensor specific to each patient, defined as $\phi_{i,j} = \begin{bmatrix} PPGR_{i,j} \\ [Leucine]_{i,j}\end{bmatrix}$, where PPGR refers to averaged postprandial glycemic response over the day. Leucine, an amino acid in particularly high abundance in dairy and meat proteins, is used as a biomarker highly correlated with the lipid/protein macronutrient proportion of a patient's diet. 

Herein, the concentration of bioavailable BPO-Clin (Benzoyl Peroxide-Clindamycin, applied topically) is modeled as a single-compartment model consisting of the sebocytes in the epidermis where the topical is applied.  

Thus, the ordinary differential equation $\frac{d}{dx}\begin{bmatrix} k_{Clin\,bioa} * k_{hydrolysis}*[Clin-Phos]_{i} - k_{elim\,Clin} * [Clin]_{SS} \\ k_{BPO\,bioa} * (v_{max,\,homolytic} * [BPO]_{i})/ ([BPO]_{i} + K_{max}) - k_{elim\,BPO} * [BPO \, \bullet]_{SS}\end{bmatrix}$ describes the rate of change of concentration in the components of the vector $\hat T_{t}$. Subject to local linearity, this modeling allows determination of the current bioactive concentrations of treatment components over the treatment series. 

$PPGR_{i,j}$ is calculated using a simplified approach to the approach covered in Zeevi et al. (2015). Here, the feature matrix $\chi_{i, j}$ is used to predict $PPGR_{i,j, k}$: $\chi_{i, j}\top = \begin{bmatrix} [Dietary\,Fiber\,(g)]_{k} & [Remaining\,Carbs\,(g)]_k & [Saturated\,Fat\,(g)]_k & [Unsaturated\,Fat\,(g)]_k & [Water\,(mL)]_k & [Sodium\,(mg)]_k & BMI\end{bmatrix}$. The index k refers to the meal index corresponding to this PPGR (ranging from 0 - 2) , such that the feature matrix is dimension 3 x 7. Here, a Gradient Boosting Regression Tree with a standard MSE loss function is used to fit the predictor. Using all three meals in the feature matrix helps motivate patients to ensure compliance in the form of eating as many optimal meals a day. 

$PPGR_{i, j}$'s components are calculated in the same way as in the above publication, as the iAUC (incremental area under the curve) of blood glucose levels 2 hours post-meal.  


### Expectation-Maximization of Model Parameters and Latent State Acne Severity Change State Distribution
 
This implementation applies EM to the joint likelihood of all layers, assuming the Markov propery holds between each. That is, the likelihood maximized is: $$\mathcal{L} = \ln Pr(Latent,\,Imputed,\,Observed)  = \ln Pr(L | I, \theta_{2}) + \ln Pr(I | O, \theta_{1}) + Const.$$

The expectation step calculates the first moments for both Y and Z, $\mathcal{E}(\hat y_t), \mathcal{E}(\hat z_t)$, as well as the second moment matrices, $\mathcal{E}(\hat y_t \hat y_t^{T}), \mathcal{E}(\hat z_t \hat z_t^{T})$, with posterior variances included. 

The cross moments between Y and Z, $\mathcal{E}(\hat z_t \hat y_t^{T})$ and $\mathcal{E}(\hat z_{t} \hat z_{t-1}^{T})$ are also found here.

EM is also applied to optimize the NLME's parameters, listed above. 



$$\textbf{Maximization of Parameters}$$

The maximization step maximizes both the weights/biases and the model parameters, that is, $\theta_{1}, \theta_{2}$, W, and A.  

In [1]:
from acne_model import data_parsing
from acne_model import data_visualization
from acne_model import model_building, process_data_and_build_HBM
import json
import numpy as np

def main(this_raw_data_name, this_diet_raw_data_name, json_name):
    data_returns = data_parsing(this_raw_data_name, this_diet_raw_data_name)
    these_frames = data_returns[0]
    this_patients_diets = data_returns[1]
    this_patients_treatments = data_returns[2]
    this_patients_iAUCs = data_returns[3]
    these_raw_frames = data_returns[4]

    
    
    this_HBM = process_data_and_build_HBM(these_frames, model_priors_and_config_handle = json_name, these_raw_frames = these_raw_frames, 
                                         patients_diet_data = this_patients_diets, patients_treatment_data = this_patients_treatments, 
                                          patients_iAUCs = this_patients_iAUCs)
    
#retrieving Acne04 dataset from Kaggle source (local download on machine)
this_raw_data_name = "sim_acne_amended_v2.csv"
this_diet_data_name = "sim_acne_diet.csv"
this_json_name = "bhm_model_config.json"
if __name__ == "__main__":
    main(this_raw_data_name, this_diet_data_name, this_json_name)

{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'lea

Citations:

Zeevi, D., et al. (2015). "Personalized Nutrition by Prediction of Glycemic Responses." Cell, 163(5), 1079-1094. doi:10.1016/j.cell.2015.11.001.


## 